## tl;dr

This notebook profiles the Phase 3 curated feature dataset for FraudOps. It checks fraud rate, amount distribution, fraud patterns by channel, merchant category, country, hour, device age, known/new device behavior, and transaction velocity. Run the notebook after `scripts/run_batch_ingestion.py` creates curated feature Parquet files.

## Context & Methods

### Key Assumptions

- The curated feature dataset is generated by the latest successful batch pipeline run.
- `fraud_label == 'fraud'` is treated as the positive class for exploration.
- Rolling features were created in `src/fraudops/features.py` using prior transactions only.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
FEATURE_ROOT = PROJECT_ROOT / 'data' / 'curated' / 'analytical_features'

feature_files = sorted(FEATURE_ROOT.rglob('features.parquet'))
assert feature_files, f'No curated feature files found under {FEATURE_ROOT}'
latest_feature_file = max(feature_files, key=lambda path: path.stat().st_mtime)
features = pd.read_parquet(latest_feature_file)
features['transaction_at'] = pd.to_datetime(features['transaction_at'], utc=True)
features['is_fraud'] = features['fraud_label'].eq('fraud')
latest_feature_file, features.shape

(WindowsPath('C:/Users/sarth/Documents/FraudOps/data/curated/analytical_features/pipeline_run_id=phase3_full_002/features.parquet'),
 (10000, 39))

## Data

In [2]:
summary = pd.DataFrame({
    'metric': [
        'rows',
        'columns',
        'min_transaction_at',
        'max_transaction_at',
        'fraud_count',
        'fraud_rate',
    ],
    'value': [
        len(features),
        features.shape[1],
        features['transaction_at'].min(),
        features['transaction_at'].max(),
        int(features['is_fraud'].sum()),
        round(float(features['is_fraud'].mean()), 4),
    ],
})
summary

,metric,value
0,rows,10000
1,columns,39
2,min_transaction_at,2026-05-14 19:47:21+00:00
3,max_transaction_at,2026-07-13 19:41:21+00:00
4,fraud_count,200
5,fraud_rate,0.02


In [3]:
features[['transaction_amount', 'customer_avg_amount_30d', 'amount_to_customer_avg_ratio', 'transactions_last_1h', 'device_age_days']].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T

,count,mean,std,min,1%,5%,50%,95%,99%,max
transaction_amount,10000.0,39.793681,69.567658,1.010000,3.159900,5.849500,24.525000,106.793500,275.328300,2184.800000
customer_avg_amount_30d,10000.0,38.656306,36.595966,2.400000,6.546467,10.509722,28.670417,95.295900,185.453618,755.780000
amount_to_customer_avg_ratio,10000.0,1.215323,2.633134,0.008092,0.117562,0.224956,0.849540,3.050357,6.706156,176.406944
transactions_last_1h,10000.0,0.013000,0.114159,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2.000000
device_age_days,10000.0,905.158776,511.162165,0.000000,0.013188,96.212847,894.613194,1689.662361,1769.819819,1819.980556


## Results

In [4]:
def fraud_rate_table(column: str, limit: int = 15) -> pd.DataFrame:
    table = (
        features.groupby(column, dropna=False)
        .agg(transactions=('transaction_id', 'count'), frauds=('is_fraud', 'sum'), fraud_rate=('is_fraud', 'mean'))
        .sort_values(['fraud_rate', 'transactions'], ascending=[False, False])
        .head(limit)
        .reset_index()
    )
    table['fraud_rate'] = table['fraud_rate'].round(4)
    return table

fraud_rate_table('fraud_scenario')

,fraud_scenario,transactions,frauds,fraud_rate
0,behavior_deviation,45,45,1.0
1,new_device_takeover,37,37,1.0
2,card_testing,32,32,1.0
3,merchant_fraud_burst,30,30,1.0
4,impossible_travel,28,28,1.0
5,transaction_velocity,28,28,1.0
6,,9700,0,0.0
7,normal_travel_false_positive,100,0,0.0


In [5]:
fraud_rate_table('is_card_present')

,is_card_present,transactions,frauds,fraud_rate
0,False,7520,168,0.0223
1,True,2480,32,0.0129


In [6]:
fraud_rate_table('is_international')

,is_international,transactions,frauds,fraud_rate
0,True,1370,28,0.0204
1,False,8630,172,0.0199


In [7]:
hourly = features.groupby('transaction_hour').agg(transactions=('transaction_id', 'count'), fraud_rate=('is_fraud', 'mean')).reset_index()
hourly['fraud_rate'] = hourly['fraud_rate'].round(4)
hourly.head(24)

,transaction_hour,transactions,fraud_rate
0,0,442,0.0181
1,1,414,0.0193
2,2,453,0.0132
3,3,371,0.0350
4,4,410,0.0098
5,5,414,0.0193
6,6,363,0.0193
7,7,413,0.0121
8,8,464,0.0172
9,9,451,0.0200


In [8]:
velocity_bins = pd.cut(features['transactions_last_1h'], bins=[-1, 0, 1, 3, 10, 1000], labels=['0', '1', '2-3', '4-10', '10+'])
velocity = features.assign(velocity_bin=velocity_bins).groupby('velocity_bin', observed=False).agg(transactions=('transaction_id', 'count'), fraud_rate=('is_fraud', 'mean')).reset_index()
velocity['fraud_rate'] = velocity['fraud_rate'].round(4)
velocity

,velocity_bin,transactions,fraud_rate
0,0,9871,0.0202
1,1,128,0.0078
2,2-3,1,0.0000
3,4-10,0,NaN
4,10+,0,NaN


In [9]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features['transaction_amount'].clip(upper=features['transaction_amount'].quantile(0.99)).hist(ax=axes[0, 0], bins=40)
axes[0, 0].set_title('Transaction amount, capped at p99')
axes[0, 0].set_xlabel('Amount')
axes[0, 0].set_ylabel('Transactions')

features.groupby('transaction_hour')['is_fraud'].mean().plot(ax=axes[0, 1], marker='o')
axes[0, 1].set_title('Fraud rate by hour')
axes[0, 1].set_xlabel('Hour')
axes[0, 1].set_ylabel('Fraud rate')

features.groupby('is_international')['is_fraud'].mean().plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('Fraud rate by international flag')
axes[1, 0].set_xlabel('Is international')
axes[1, 0].set_ylabel('Fraud rate')

features['transactions_last_1h'].clip(upper=10).hist(ax=axes[1, 1], bins=11)
axes[1, 1].set_title('Transactions in prior hour, capped at 10')
axes[1, 1].set_xlabel('Prior-hour count')
axes[1, 1].set_ylabel('Transactions')

plt.tight_layout()

## Takeaways

Use the executed tables above to identify which synthetic fraud patterns are easiest to separate from normal behavior. Before model training, pay special attention to normal-travel false positives, new-device transactions, and velocity-heavy scenarios because they stress the balance between fraud capture and customer friction.